# Filtering the IMDb Class Database (2010+, English/US Titles)

This notebook walks you through:
1. Connecting to the full `imdb_class.db` (203 MB, ~57K titles)
2. Exploring the schema and data
3. Filtering to **2010–present**, **English-language** titles familiar to US students
4. Pruning the related tables (ratings, principals, crew, names)
5. **Removing titles by prolific non-US actor names** (Indian cinema, etc.)
6. **Removing remaining non-US titles by cast composition analysis** (30%+ South Asian actors)
7. Verifying and exporting the filtered result as a new `.db` file

> **Why filter?** The full database spans 1906–2025 and includes many foreign-language films. For classroom use, a smaller database focused on recent, recognizable titles makes queries more engaging and relatable.

In [19]:
import sqlite3
import os

# Path to the FULL database (203 MB)
SOURCE_DB = "imdb_class.db"

# Make sure the file exists and isn't empty
size_mb = os.path.getsize(SOURCE_DB) / (1024 * 1024)
print(f"Source database: {SOURCE_DB}")
print(f"File size: {size_mb:.1f} MB")
assert size_mb > 1, "ERROR: This file is empty or corrupt. Use the copy from dbApps06 or teacher/databasesDatasets."

Source database: imdb_class.db
File size: 202.4 MB


## Step 1: Connect & Explore the Schema

Let's see what tables and columns we're working with.

In [20]:
conn = sqlite3.connect(SOURCE_DB)
cursor = conn.cursor()

# List all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = cursor.fetchall()

for (table_name,) in tables:
    cursor.execute(f"PRAGMA table_info({table_name})")
    cols = cursor.fetchall()
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    print(f"\n{table_name}  ({count:,} rows)")
    print("-" * 40)
    for col in cols:
        print(f"  {col[1]:25s} {col[2]}")


name_basics  (400,017 rows)
----------------------------------------
  nconst                    TEXT
  primaryName               TEXT
  birthYear                 INTEGER
  deathYear                 INTEGER
  primaryProfession         TEXT
  knownForTitles            TEXT

title_basics  (57,198 rows)
----------------------------------------
  tconst                    TEXT
  titleType                 TEXT
  primaryTitle              TEXT
  originalTitle             TEXT
  isAdult                   INTEGER
  startYear                 INTEGER
  endYear                   INTEGER
  runtimeMinutes            INTEGER
  genres                    TEXT

title_crew_person  (354,189 rows)
----------------------------------------
  tconst                    TEXT
  nconst                    TEXT
  role                      TEXT

title_principals  (1,084,764 rows)
----------------------------------------
  tconst                    TEXT
  ordering                  INTEGER
  nconst                  

## Step 2: Understand the Data

Let's look at the year distribution and how `primaryTitle` vs `originalTitle` can help us identify English-language titles.

In [21]:
import pandas as pd

# Year distribution
df = pd.read_sql_query('''
    SELECT startYear, COUNT(*) as count 
    FROM title_basics 
    WHERE startYear IS NOT NULL 
    GROUP BY startYear 
    ORDER BY startYear
''', conn)

print("Year range:", df['startYear'].min(), "to", df['startYear'].max())
print(f"\nTitles 2010 and later: {df[df['startYear'] >= 2010]['count'].sum():,}")
print(f"Titles before 2010:    {df[df['startYear'] < 2010]['count'].sum():,}")

Year range: 1906 to 2025

Titles 2010 and later: 29,608
Titles before 2010:    27,590


### How do we identify English / US titles?

This database doesn't have a `region` or `language` column. However, IMDb's `primaryTitle` is the **US/English market title**, while `originalTitle` is the title in its original language.

**Key insight:** If `primaryTitle == originalTitle`, the film was almost certainly made in English. Foreign films will have different original titles (e.g., *primaryTitle* = "Parasite" but *originalTitle* = "기생충").

This isn't 100% perfect — some foreign English-titled films will slip through, and a few dubbed titles might match — but it's an excellent filter for classroom purposes.

In [22]:
# How many titles match vs. don't match?
cursor.execute('''
    SELECT 
        CASE WHEN primaryTitle = originalTitle THEN 'English (titles match)' 
             ELSE 'Foreign (titles differ)' END as language_guess,
        COUNT(*) as count
    FROM title_basics
    WHERE startYear >= 2010
    GROUP BY language_guess
''')

for row in cursor.fetchall():
    print(f"  {row[0]:35s} {row[1]:,}")

print()

# Show some examples of filtered-OUT foreign titles
cursor.execute('''
    SELECT primaryTitle, originalTitle, startYear 
    FROM title_basics 
    WHERE startYear >= 2010 
      AND primaryTitle != originalTitle 
    ORDER BY RANDOM() 
    LIMIT 8
''')
print("Examples of foreign titles that WILL be filtered out:")
print("-" * 60)
for row in cursor.fetchall():
    print(f"  '{row[0]}' (original: '{row[1]}', {row[2]})")

  English (titles match)              21,937
  Foreign (titles differ)             7,671

Examples of foreign titles that WILL be filtered out:
------------------------------------------------------------
  'Love Untangled' (original: 'Gobaekeui yeoksa', 2025)
  'There's No Place Like Home' (original: 'A casa tutti bene', 2018)
  'A Returner's Magic Should Be Special' (original: 'Kikansha no Mahou wa Tokubetsu desu', 2023)
  'Special Delivery' (original: 'Teuksong', 2022)
  '[REC] 3: Genesis' (original: '[Rec]³: Génesis', 2012)
  'Afire' (original: 'Roter Himmel', 2023)
  'Cathedral of the Sea' (original: 'La catedral del mar', 2018)
  'Nobody Sleeps in the Woods Tonight 2' (original: 'W lesie dzis nie zasnie nikt 2', 2021)


In [23]:
# What types of titles are in the 2010+ English set?
cursor.execute('''
    SELECT titleType, COUNT(*) as count
    FROM title_basics
    WHERE startYear >= 2010
      AND primaryTitle = originalTitle
      AND isAdult = 0
    GROUP BY titleType
    ORDER BY count DESC
''')

print("Title types in our filtered set (2010+, English, non-adult):")
print("-" * 50)
for row in cursor.fetchall():
    print(f"  {row[0]:15s} {row[1]:,}")

Title types in our filtered set (2010+, English, non-adult):
--------------------------------------------------
  movie           16,889
  tvSeries        5,048


## Step 3: Build the Filtered Database

Now we'll create a new database with only:
- **Titles** from 2010 onward where `primaryTitle == originalTitle` (English proxy) and `isAdult = 0`
- **Ratings** for those titles only
- **Principals** (actors, directors, etc.) for those titles only  
- **Crew** for those titles only
- **Names** for people who appear in the filtered titles

In [24]:
# Output file name
OUTPUT_DB = "imdb_class_2010plus.db"

# Remove old file if it exists
if os.path.exists(OUTPUT_DB):
    os.remove(OUTPUT_DB)

new_conn = sqlite3.connect(OUTPUT_DB)
new_cursor = new_conn.cursor()

print("Creating filtered database...")

Creating filtered database...


In [25]:
# 1. Filter title_basics
new_cursor.execute('''CREATE TABLE title_basics (
    tconst TEXT PRIMARY KEY,
    titleType TEXT,
    primaryTitle TEXT,
    originalTitle TEXT,
    isAdult INTEGER,
    startYear INTEGER,
    endYear INTEGER,
    runtimeMinutes INTEGER,
    genres TEXT
)''')

cursor.execute('''
    SELECT * FROM title_basics
    WHERE startYear >= 2010
      AND primaryTitle = originalTitle
      AND isAdult = 0
''')
rows = cursor.fetchall()
new_cursor.executemany('INSERT INTO title_basics VALUES (?,?,?,?,?,?,?,?,?)', rows)
new_conn.commit()

print(f"title_basics: {len(rows):,} titles (from 57,198 original)")

title_basics: 21,937 titles (from 57,198 original)


In [26]:
# 2. Filter title_ratings (only titles we kept)
new_cursor.execute('''CREATE TABLE title_ratings (
    tconst TEXT PRIMARY KEY,
    averageRating REAL,
    numVotes INTEGER
)''')

cursor.execute('''
    SELECT r.* FROM title_ratings r
    INNER JOIN title_basics b ON r.tconst = b.tconst
    WHERE b.startYear >= 2010
      AND b.primaryTitle = b.originalTitle
      AND b.isAdult = 0
''')
rows = cursor.fetchall()
new_cursor.executemany('INSERT INTO title_ratings VALUES (?,?,?)', rows)
new_conn.commit()

print(f"title_ratings: {len(rows):,} rows")

title_ratings: 21,937 rows


In [27]:
# 3. Filter title_principals
new_cursor.execute('''CREATE TABLE title_principals (
    tconst TEXT,
    ordering INTEGER,
    nconst TEXT,
    category TEXT,
    job TEXT,
    characters TEXT
)''')

cursor.execute('''
    SELECT p.* FROM title_principals p
    INNER JOIN title_basics b ON p.tconst = b.tconst
    WHERE b.startYear >= 2010
      AND b.primaryTitle = b.originalTitle
      AND b.isAdult = 0
''')
rows = cursor.fetchall()
new_cursor.executemany('INSERT INTO title_principals VALUES (?,?,?,?,?,?)', rows)
new_conn.commit()

print(f"title_principals: {len(rows):,} rows")

title_principals: 412,753 rows


In [28]:
# 4. Filter title_crew_person
new_cursor.execute('''CREATE TABLE title_crew_person (
    tconst TEXT,
    nconst TEXT,
    role TEXT
)''')

cursor.execute('''
    SELECT c.* FROM title_crew_person c
    INNER JOIN title_basics b ON c.tconst = b.tconst
    WHERE b.startYear >= 2010
      AND b.primaryTitle = b.originalTitle
      AND b.isAdult = 0
''')
rows = cursor.fetchall()
new_cursor.executemany('INSERT INTO title_crew_person VALUES (?,?,?)', rows)
new_conn.commit()

print(f"title_crew_person: {len(rows):,} rows")

title_crew_person: 126,450 rows


In [29]:
# 5. Filter name_basics — only people connected to our filtered titles
# Collect all nconst values from principals + crew
new_cursor.execute('SELECT DISTINCT nconst FROM title_principals')
principal_ids = set(row[0] for row in new_cursor.fetchall())

new_cursor.execute('SELECT DISTINCT nconst FROM title_crew_person')
crew_ids = set(row[0] for row in new_cursor.fetchall())

all_person_ids = principal_ids | crew_ids
print(f"Unique people referenced in filtered titles: {len(all_person_ids):,}")

# Create the table and insert matching names
new_cursor.execute('''CREATE TABLE name_basics (
    nconst TEXT PRIMARY KEY,
    primaryName TEXT,
    birthYear INTEGER,
    deathYear INTEGER,
    primaryProfession TEXT,
    knownForTitles TEXT
)''')

# Fetch in batches (SQLite has a variable limit)
batch_size = 500
person_list = list(all_person_ids)
inserted = 0

for i in range(0, len(person_list), batch_size):
    batch = person_list[i:i+batch_size]
    placeholders = ','.join(['?'] * len(batch))
    cursor.execute(f'SELECT * FROM name_basics WHERE nconst IN ({placeholders})', batch)
    rows = cursor.fetchall()
    if rows:
        new_cursor.executemany('INSERT INTO name_basics VALUES (?,?,?,?,?,?)', rows)
        inserted += len(rows)

new_conn.commit()
print(f"name_basics: {inserted:,} people (from 400,017 original)")

Unique people referenced in filtered titles: 194,097
name_basics: 194,097 people (from 400,017 original)


## Step 4: Add Indexes for Performance

Indexes make JOINs and lookups much faster — important for classroom queries.

In [30]:
# Add useful indexes
indexes = [
    "CREATE INDEX idx_principals_tconst ON title_principals(tconst)",
    "CREATE INDEX idx_principals_nconst ON title_principals(nconst)",
    "CREATE INDEX idx_crew_tconst ON title_crew_person(tconst)",
    "CREATE INDEX idx_crew_nconst ON title_crew_person(nconst)",
    "CREATE INDEX idx_basics_year ON title_basics(startYear)",
    "CREATE INDEX idx_basics_type ON title_basics(titleType)",
]

for idx_sql in indexes:
    new_cursor.execute(idx_sql)
    print(f"Created: {idx_sql.split('CREATE INDEX ')[1].split(' ON')[0]}")

new_conn.commit()

Created: idx_principals_tconst
Created: idx_principals_nconst
Created: idx_crew_tconst
Created: idx_crew_nconst
Created: idx_basics_year
Created: idx_basics_type


## Step 5: Remove Non-US Film Actors and Their Titles

The `primaryTitle == originalTitle` filter catches most foreign films, but many Indian (Bollywood/Tollywood) films use English titles natively. We can clean these out by identifying prolific Indian-cinema actors and removing all titles they appear in, then pruning orphaned names.

> **Note:** You can edit the `actors_to_remove` list below to add or remove names as needed.

In [31]:
# Prolific non-US actors whose titles we want to remove
actors_to_remove = [
    # Pass 1: highest-count Indian cinema actors
    'Nassar', 'Vennela Kishore', 'Prakash Raj', 'Tanikella Bharani',
    'Yogi Babu', 'Brahmanandam', 'Siddique', 'Murli Sharma',
    'Akshay Kumar', 'V. Jayaprakash', 'Ajay', 'Rao Ramesh',
    'Anupam Kher', 'Krishna Murali Posani', 'Mohammad Ali',
    # Pass 2: additional Indian cinema actors
    'Rajesh Sharma', 'Sanjay Mishra', 'Nawazuddin Siddiqui',
    'Ajay Devgn', 'Vijay Raaz', 'Prithviraj Sukumaran',
    'Achyuth Kumar', 'Aju Varghese', 'Tovino Thomas', 'Rajkummar Rao', 'Sita Ramam', 'Soorarai Pottru', 
]

# Find their nconst IDs
placeholders = ','.join(['?'] * len(actors_to_remove))
new_cursor.execute(
    f"SELECT nconst, primaryName FROM name_basics WHERE primaryName IN ({placeholders})",
    actors_to_remove
)
actor_rows = new_cursor.fetchall()
actor_ids = [r[0] for r in actor_rows]

print(f"Found {len(actor_rows)} matching actor records for {len(actors_to_remove)} names:")
for nconst, name in actor_rows:
    print(f"  {nconst}: {name}")

Found 40 matching actor records for 27 names:
  nm1302330: Murli Sharma
  nm2496992: Ajay
  nm3559559: Rao Ramesh
  nm1578310: Ajay
  nm13122537: Ajay
  nm0103977: Brahmanandam
  nm10805786: Ajay
  nm4331023: Rajesh Sharma
  nm0474774: Akshay Kumar
  nm1596350: Nawazuddin Siddiqui
  nm0019382: Mohammad Ali
  nm0621937: Nassar
  nm1388202: Siddique
  nm0695177: Prakash Raj
  nm4028151: Aju Varghese
  nm12052233: Ajay
  nm5731566: Rajesh Sharma
  nm11459719: Ajay
  nm12567685: Akshay Kumar
  nm0592799: Sanjay Mishra
  nm13982496: Ajay
  nm1335387: Prithviraj Sukumaran
  nm0788903: Rajesh Sharma
  nm0080238: Tanikella Bharani
  nm15927318: Ajay
  nm3822770: Rajkummar Rao
  nm3808948: Vennela Kishore
  nm0692586: Krishna Murali Posani
  nm13537093: Ajay
  nm1957308: V. Jayaprakash
  nm17868242: Ajay
  nm5724719: Achyuth Kumar
  nm4289673: Akshay Kumar
  nm0796504: Siddique
  nm0451600: Anupam Kher
  nm5732707: Tovino Thomas
  nm0704694: Vijay Raaz
  nm13025207: Ajay
  nm6489058: Yogi Babu


In [32]:
# Find all titles these actors appear in
id_placeholders = ','.join(['?'] * len(actor_ids))
new_cursor.execute(
    f"SELECT DISTINCT tconst FROM title_principals WHERE nconst IN ({id_placeholders})",
    actor_ids
)
titles_to_remove = [r[0] for r in new_cursor.fetchall()]
print(f"Titles connected to these actors: {len(titles_to_remove)}")

# Show before counts
print("\n=== BEFORE ACTOR REMOVAL ===")
for table in ['title_basics', 'title_ratings', 'title_principals', 'title_crew_person', 'name_basics']:
    new_cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"  {table:25s} {new_cursor.fetchone()[0]:>10,} rows")

Titles connected to these actors: 1097

=== BEFORE ACTOR REMOVAL ===
  title_basics                  21,937 rows
  title_ratings                 21,937 rows
  title_principals             412,753 rows
  title_crew_person            126,450 rows
  name_basics                  194,097 rows


In [33]:
# Delete titles from all title-based tables
title_placeholders = ','.join(['?'] * len(titles_to_remove))

for table in ['title_basics', 'title_ratings', 'title_principals', 'title_crew_person']:
    new_cursor.execute(f"DELETE FROM {table} WHERE tconst IN ({title_placeholders})", titles_to_remove)
    print(f"  Deleted {new_cursor.rowcount:,} rows from {table}")

new_conn.commit()

# Prune orphaned names (people no longer connected to any remaining title)
new_cursor.execute("""
    DELETE FROM name_basics 
    WHERE nconst NOT IN (SELECT DISTINCT nconst FROM title_principals)
      AND nconst NOT IN (SELECT DISTINCT nconst FROM title_crew_person)
""")
print(f"  Pruned {new_cursor.rowcount:,} orphaned names from name_basics")
new_conn.commit()

# Show after counts
print("\n=== AFTER ACTOR REMOVAL ===")
for table in ['title_basics', 'title_ratings', 'title_principals', 'title_crew_person', 'name_basics']:
    new_cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"  {table:25s} {new_cursor.fetchone()[0]:>10,} rows")

  Deleted 1,097 rows from title_basics
  Deleted 1,097 rows from title_ratings
  Deleted 21,953 rows from title_principals
  Deleted 3,928 rows from title_crew_person
  Pruned 3,511 orphaned names from name_basics

=== AFTER ACTOR REMOVAL ===
  title_basics                  20,840 rows
  title_ratings                 20,840 rows
  title_principals             390,800 rows
  title_crew_person            122,522 rows
  name_basics                  190,586 rows


## Step 6: Remove Remaining Indian Titles by Cast Analysis

The actor-name removal above catches the most prolific actors, but many Indian films remain because their casts weren't on our explicit list. This step scans **every remaining title** and flags it for removal if 30%+ of its actors have South Asian name patterns.

We protect known Hollywood films that happen to have Indian-origin actors (e.g., *Life of Pi*, *The Man Who Knew Infinity*, *Yesterday*) using a false-positives list.

> **Note:** You can adjust the `SA_THRESHOLD` (default 0.3) or add to the `FALSE_POSITIVES` set as needed.

In [34]:
# --- Configuration ---
SA_THRESHOLD = 0.30   # Flag title if >= 30% of actors match South Asian name patterns
MIN_ACTORS   = 3      # Need at least 3 actors to make a reliable judgment

# South Asian name-component indicators (lowercase substrings)
SA_NAME_INDICATORS = [
    # Common surnames / name components
    'kumar', 'sharma', 'singh', 'patel', 'khan', 'babu', 'rao', 'raju',
    'reddy', 'nair', 'menon', 'pillai', 'iyer', 'mukherjee', 'chatterjee',
    'banerjee', 'dasgupta', 'ghosh', 'dutta', 'bose',
    'srinivasan', 'subramaniam', 'krishnan', 'raghavan', 'venkatesh',
    'pandey', 'mishra', 'gupta', 'verma', 'joshi', 'kadam', 'patil',
    'kulkarni', 'deshmukh', 'jadhav', 'bhat', 'hegde', 'gowda',
    'srinath', 'prasad', 'murthy', 'swamy', 'acharya', 'kamath', 'shenoy',
    'bhasi', 'sahir', 'dutt', 'khanna', 'kapoor', 'advani', 'thakur',
    'mandanna', 'prabhu', 'shetty', 'talpade',
    # Well-known Indian film stars (first or full names)
    'sethupathi', 'faasil', 'mammootty', 'mohanlal', 'suriya', 'dhanush',
    'kamal haasan', 'rajinikanth', 'chiranjeevi', 'prabhas',
    'allu arjun', 'ram charan', 'dulquer', 'nivin pauly', 'fahadh',
    'sai pallavi', 'samantha ruth', 'samantha', 'karthi',
    'aparna', 'balamurali', 'kaushal', 'malhotra', 'khurrana', 'rajput',
    'madhavan', 'ranveer', 'paresh rawal', 'tabu', 'irrfan', 'nawaz',
    'pankaj', 'manoj', 'rajit', 'sachin', 'arjun', 'varun', 'ranbir',
    'alia', 'deepika', 'priyanka', 'kangana', 'hrithik', 'shahid',
    'ayushmann', 'rajkummar', 'bhumi', 'kriti', 'rashmika', 'vijay',
    'diljit', 'tiger', 'shraddha', 'nawazuddin', 'parvathy', 'nazriya',
    'tovino', 'kiara', 'sidharth', 'vicky',
]

# Hollywood films to protect — they have Indian-origin actors but are US/intl productions
FALSE_POSITIVES = {
    'tt0454876',   # Life of Pi
    'tt1375666',   # Inception
    'tt0816692',   # Interstellar
    'tt1345836',   # The Dark Knight Rises
    'tt4154756',   # Avengers: Infinity War
    'tt4154796',   # Avengers: Endgame
    'tt9362722',   # Spider-Man: Into the Spider-Verse
    'tt4633694',   # Spider-Man: Across the Spider-Verse
    'tt10872600',  # Spider-Man: No Way Home
    'tt6320628',   # Spider-Man: Far From Home
    'tt2250912',   # Spider-Man: Homecoming
    'tt1454468',   # Gravity
    'tt1596363',   # The Big Short
    'tt5108870',   # Morbius
    'tt6723592',   # Tenet
    'tt8079248',   # Yesterday (Danny Boyle)
    'tt9243804',   # The Green Knight
    'tt7286456',   # Joker
    'tt1517268',   # Barbie
    'tt15398776',  # Oppenheimer
    'tt0770828',   # Man of Steel
    'tt2015381',   # Guardians of the Galaxy
    'tt3896198',   # Guardians of the Galaxy Vol. 2
    'tt6791350',   # Guardians of the Galaxy Vol. 3
    'tt1877830',   # The Batman
    'tt1160419',   # Dune
    'tt15239678',  # Dune: Part Two
    'tt2911666',   # John Wick
    'tt1431045',   # Deadpool
    'tt5052448',   # Get Out
    'tt0478970',   # Ant-Man
    'tt4912910',   # Mission: Impossible - Fallout
    'tt9603212',   # Mission: Impossible - Dead Reckoning
    'tt8178634',   # Rocketman
    'tt5541848',   # What the Health
    'tt1424432',   # Senna
    'tt0787524',   # The Man Who Knew Infinity
}

print(f"Configured: {len(SA_NAME_INDICATORS)} name indicators, {len(FALSE_POSITIVES)} protected titles")
print(f"Threshold: flag if >= {SA_THRESHOLD:.0%} of actors match")

Configured: 116 name indicators, 37 protected titles
Threshold: flag if >= 30% of actors match


In [35]:
# Scan every title: get actors, check what % match SA name patterns
new_cursor.execute('''
    SELECT b.tconst, b.primaryTitle, b.startYear, b.titleType,
           GROUP_CONCAT(DISTINCT n.primaryName) as actors
    FROM title_basics b
    LEFT JOIN title_principals p ON b.tconst = p.tconst 
        AND p.category IN ('actor','actress')
    LEFT JOIN name_basics n ON p.nconst = n.nconst
    GROUP BY b.tconst
''')

indian_tconsts = []
for row in new_cursor.fetchall():
    tconst, title, year, ttype, actors = row
    if not actors or tconst in FALSE_POSITIVES:
        continue
    actor_list = [a.strip() for a in actors.split(',')]
    if len(actor_list) < MIN_ACTORS:
        continue
    
    sa_count = sum(
        1 for a in actor_list
        if any(ind in a.lower() for ind in SA_NAME_INDICATORS)
    )
    ratio = sa_count / len(actor_list)
    
    if ratio >= SA_THRESHOLD:
        indian_tconsts.append(tconst)

print(f"Titles flagged by cast analysis: {len(indian_tconsts)}")

# Show before counts
print("\n=== BEFORE CAST-BASED REMOVAL ===")
for table in ['title_basics', 'title_ratings', 'title_principals', 'title_crew_person', 'name_basics']:
    new_cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"  {table:25s} {new_cursor.fetchone()[0]:>10,} rows")

Titles flagged by cast analysis: 1378

=== BEFORE CAST-BASED REMOVAL ===
  title_basics                  20,840 rows
  title_ratings                 20,840 rows
  title_principals             390,800 rows
  title_crew_person            122,522 rows
  name_basics                  190,586 rows


In [36]:
# Remove flagged titles and cascade through all tables
if indian_tconsts:
    t_ph = ','.join(['?'] * len(indian_tconsts))
    
    for table in ['title_basics', 'title_ratings', 'title_principals', 'title_crew_person']:
        new_cursor.execute(f"DELETE FROM {table} WHERE tconst IN ({t_ph})", indian_tconsts)
        print(f"  Deleted {new_cursor.rowcount:,} rows from {table}")
    new_conn.commit()

    # Prune orphaned names
    new_cursor.execute("""
        DELETE FROM name_basics 
        WHERE nconst NOT IN (SELECT DISTINCT nconst FROM title_principals)
          AND nconst NOT IN (SELECT DISTINCT nconst FROM title_crew_person)
    """)
    print(f"  Pruned {new_cursor.rowcount:,} orphaned names from name_basics")
    new_conn.commit()
else:
    print("  No additional titles to remove.")

# Show after counts
print("\n=== AFTER CAST-BASED REMOVAL ===")
for table in ['title_basics', 'title_ratings', 'title_principals', 'title_crew_person', 'name_basics']:
    new_cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"  {table:25s} {new_cursor.fetchone()[0]:>10,} rows")

  Deleted 1,378 rows from title_basics
  Deleted 1,378 rows from title_ratings
  Deleted 25,952 rows from title_principals
  Deleted 5,552 rows from title_crew_person
  Pruned 8,571 orphaned names from name_basics

=== AFTER CAST-BASED REMOVAL ===
  title_basics                  19,462 rows
  title_ratings                 19,462 rows
  title_principals             364,848 rows
  title_crew_person            116,970 rows
  name_basics                  182,015 rows


## Step 7: Verify the Filtered Database

Let's confirm everything looks right after all filtering.

In [37]:
# Summary of the new database
print("=" * 55)
print("  FILTERED DATABASE SUMMARY")
print("=" * 55)

new_cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
for (table_name,) in new_cursor.fetchall():
    new_cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = new_cursor.fetchone()[0]
    print(f"  {table_name:25s} {count:>10,} rows")

file_size = os.path.getsize(OUTPUT_DB) / (1024 * 1024)
print(f"\n  File size: {file_size:.1f} MB (down from {os.path.getsize(SOURCE_DB)/1024/1024:.1f} MB)")
print("=" * 55)

  FILTERED DATABASE SUMMARY
  name_basics                  182,015 rows
  title_basics                  19,462 rows
  title_crew_person            116,970 rows
  title_principals             364,848 rows
  title_ratings                 19,462 rows

  File size: 67.6 MB (down from 202.4 MB)


In [38]:
# Spot check: top-rated recent movies your students will recognize
query = '''
    SELECT b.primaryTitle, b.startYear, b.genres, 
           r.averageRating, r.numVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND r.numVotes > 50000
    ORDER BY r.averageRating DESC
    LIMIT 20
'''

df_check = pd.read_sql_query(query, new_conn)
print("Top-rated popular movies in the filtered database:")
print("-" * 70)
print(df_check.to_string(index=False))

Top-rated popular movies in the filtered database:
----------------------------------------------------------------------
                       primaryTitle  startYear                      genres  averageRating  numVotes
                          Inception       2010     Action,Adventure,Sci-Fi            8.8   2767665
                       Interstellar       2014      Adventure,Drama,Sci-Fi            8.7   2453389
                   Django Unchained       2012               Drama,Western            8.5   1850242
                           Whiplash       2014                 Drama,Music            8.5   1124940
Spider-Man: Across the Spider-Verse       2023  Action,Adventure,Animation            8.5    499492
              The Dark Knight Rises       2012          Action,Crime,Drama            8.4   1964816
                              Senna       2010 Biography,Documentary,Sport            8.4     83165
                     Dune: Part Two       2024      Action,Adventure,Drama    

In [39]:
# Spot check: recognizable actors
query = '''
    SELECT n.primaryName, COUNT(DISTINCT p.tconst) as title_count
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    WHERE p.category IN ('actor', 'actress')
    GROUP BY n.nconst
    ORDER BY title_count DESC
    LIMIT 20
'''

df_actors = pd.read_sql_query(query, new_conn)
print("Most frequently appearing actors/actresses:")
print("-" * 45)
print(df_actors.to_string(index=False))

Most frequently appearing actors/actresses:
---------------------------------------------
      primaryName  title_count
     J.K. Simmons           53
     Nicolas Cage           52
     Eric Roberts           52
     Bruce Willis           50
      Danny Trejo           50
     Frank Grillo           46
  Fred Tatasciore           46
Samuel L. Jackson           45
  Dermot Mulroney           45
     Grey DeLisle           44
  Michael Shannon           44
       Judy Greer           42
     Willem Dafoe           41
   Dolph Lundgren           39
  Woody Harrelson           39
Dee Bradley Baker           39
     James Franco           38
     Maya Rudolph           38
      Liam Neeson           37
       Idris Elba           37


## Step 8: Done!

The filtered database has been saved. You can now use `imdb_class_2010plus.db` in your lessons.

In [40]:
# Compact the database file and close connections
new_cursor.execute("VACUUM")
new_conn.close()
conn.close()

print(f"Filtered database saved as: {OUTPUT_DB}")
print(f"File size: {os.path.getsize(OUTPUT_DB) / (1024*1024):.1f} MB")
print()
print("You're all set! Copy this .db file to your lesson folders as needed.")

Filtered database saved as: imdb_class_2010plus.db
File size: 61.0 MB

You're all set! Copy this .db file to your lesson folders as needed.
